# Explorando Dados Amazon Sale Report

In [2]:
import pandas as pd

In [ ]:
# Carregando arquivo CSV
df_amazon = pd.read_csv(r'D:\Area de trabalho\Data-Science-Learning\01-SQL\Base de dados\Amazon Sale Report.csv')

C:\Users\w10\AppData\Local\Temp\ipykernel_19452\2221286087.py:1: DtypeWarning: Columns (0: Unnamed: 22) have mixed types. Specify dtype option on import or set low_memory=False.
  df_amazon = pd.read_csv(r'D:\Area de trabalho\Data-Science-Learning\01-SQL\Base de dados\Amazon Sale Report.csv')


In [ ]:
df_amazon.shape # Total de Linhas e Colunas

(128975, 24)

In [5]:
df_amazon.columns.tolist()

['index',
 'Order ID',
 'Date',
 'Status',
 'Fulfilment',
 'Sales Channel ',
 'ship-service-level',
 'Style',
 'SKU',
 'Category',
 'Size',
 'ASIN',
 'Courier Status',
 'Qty',
 'currency',
 'Amount',
 'ship-city',
 'ship-state',
 'ship-postal-code',
 'ship-country',
 'promotion-ids',
 'B2B',
 'fulfilled-by',
 'Unnamed: 22']

In [6]:
df_amazon.head()

,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,...,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,...,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,...,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,...,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN
3,3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,...,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,Easy Ship,NaN
4,4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,...,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,NaN,NaN


In [7]:
df_amazon.isnull().sum() # Verifica valores nulos

index                     0
Order ID                  0
Date                      0
Status                    0
Fulfilment                0
Sales Channel             0
ship-service-level        0
Style                     0
SKU                       0
Category                  0
Size                      0
ASIN                      0
Courier Status         6872
Qty                       0
currency               7795
Amount                 7795
ship-city                33
ship-state               33
ship-postal-code         33
ship-country             33
promotion-ids         49153
B2B                       0
fulfilled-by          89698
Unnamed: 22           49050
dtype: int64

In [ ]:
df_amazon.dtypes # Verifica tipo das colunas

index                   int64
Order ID                  str
Date                      str
Status                    str
Fulfilment                str
Sales Channel             str
ship-service-level        str
Style                     str
SKU                       str
Category                  str
Size                      str
ASIN                      str
Courier Status            str
Qty                     int64
currency                  str
Amount                float64
ship-city                 str
ship-state                str
ship-postal-code      float64
ship-country              str
promotion-ids             str
B2B                      bool
fulfilled-by              str
Unnamed: 22            object
dtype: object

In [ ]:
df_amazon['Amount'].describe() # Estatísticas básicas do Amount

count    121180.000000
mean        648.561465
std         281.211687
min           0.000000
25%         449.000000
50%         605.000000
75%         788.000000
max        5584.000000
Name: Amount, dtype: float64

#Limpeza dos Dados
> Remoção de colunas irrelevantes, tratamento de valores nulos e conversão de tipos.

In [10]:
df_amazon = df_amazon.drop(columns=['Unnamed: 22', 'fulfilled-by', 'promotion-ids']) # Remove as colunas que não vou utilizar

In [11]:
df_amazon = df_amazon.dropna(subset=['Amount']) #Remover linhas sem Amount (essencial para KPIs)

In [12]:
df_amazon['Courier Status'] = df_amazon['Courier Status'].fillna('Unknown') # Preencher Courier Status nulo

In [13]:
df_amazon = df_amazon.dropna(subset=['ship-city', 'ship-state']) #Remove as linhas sem localização

In [14]:
df_amazon['Date'] = pd.to_datetime(df_amazon['Date']) # Converter Date para datetime

C:\Users\w10\AppData\Local\Temp\ipykernel_19452\3563497527.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_amazon['Date'] = pd.to_datetime(df_amazon['Date']) # Converter Date para datetime


In [16]:
# Verificar resultado
print(df_amazon.shape)
df_amazon.isnull().sum()

(121149, 21)


index                 0
Order ID              0
Date                  0
Status                0
Fulfilment            0
Sales Channel         0
ship-service-level    0
Style                 0
SKU                   0
Category              0
Size                  0
ASIN                  0
Courier Status        0
Qty                   0
currency              0
Amount                0
ship-city             0
ship-state            0
ship-postal-code      0
ship-country          0
B2B                   0
dtype: int64

In [23]:
# Gera o CREATE TABLE baseado no Amazon Sales
# Limpar nomes das colunas
df_amazon.columns = df_amazon.columns.str.strip()

# Gerar CREATE TABLE correto
def gerar_create_table(df, nome_tabela):
    tipos = {
        'int64': 'INT',
        'float64': 'DECIMAL(10,2)',
        'bool': 'VARCHAR(10)',
        'object': 'VARCHAR(100)',
        'datetime64[ns]': 'DATE'
    }
    
    colunas = []
    for col, dtype in df.dtypes.items():
        tipo_sql = tipos.get(str(dtype), 'VARCHAR(100)')
        col_limpa = col.lower().replace(' ', '_').replace('-', '_')
        colunas.append(f"    {col_limpa} {tipo_sql}")
    
    sql = f"CREATE TABLE {nome_tabela} (\n"
    sql += ",\n".join(colunas)
    sql += "\n);"
    print(sql)

gerar_create_table(df_amazon, 'vendas')

CREATE TABLE vendas (
    index INT,
    order_id VARCHAR(100),
    date VARCHAR(100),
    status VARCHAR(100),
    fulfilment VARCHAR(100),
    sales_channel VARCHAR(100),
    ship_service_level VARCHAR(100),
    style VARCHAR(100),
    sku VARCHAR(100),
    category VARCHAR(100),
    size VARCHAR(100),
    asin VARCHAR(100),
    courier_status VARCHAR(100),
    qty INT,
    currency VARCHAR(100),
    amount DECIMAL(10,2),
    ship_city VARCHAR(100),
    ship_state VARCHAR(100),
    ship_postal_code DECIMAL(10,2),
    ship_country VARCHAR(100),
    b2b VARCHAR(10)
);


In [21]:
df_amazon.columns = df_amazon.columns.str.strip() # Remover espaços extras dos nomes das colunas
print(df_amazon.columns.tolist())

['index', 'Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'Qty', 'currency', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code', 'ship-country', 'B2B']


## Exportar para CSV limpo

In [17]:
df_amazon.to_csv(r'D:\Area de trabalho\Data-Science-Learning\01-SQL\base de dados\amazon_limpo.csv', index=False)

In [22]:
df_amazon.to_csv(r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\amazon_limpo.csv', index=False) # Local para ser salvo o aqrquivo CVS aonde irei consumir os dados em SQL